# Interactive Notebook 05 - PMSM operation point selection:

This interactive Jupyter notebook introduces the concept of operating point selection for permanent magnet synchronous motors.

For help with the installation of the required software, consider the comments in ```CTPD_course\interactive_notebooks\README.md```.
Throughout the exercises, we will be using a combination of scientific computation libraries from the [JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) ecosystem and visualize them with [matplotlib](https://matplotlib.org/) and [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/).

### Preliminaries & Imports:

In [ ]:
# automatically reloads imported ```.py```-files once they are changed and saved
%load_ext autoreload
%autoreload 2

In [ ]:
%%html
<style>
div.jupyter-widgets.widget-label {display: none;}
</style>

In [ ]:
# imports required packages
from functools import partial
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from scipy.io import loadmat
from pathlib import Path
from matplotlib import rc
import numpy as np
mpl.rcParams.update({'font.size': 20})

import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import jaxopt
import equinox as eqx
import diffrax

(**Optional**: If you have LaTeX installed, you can use the following lines for pretty rendering of plot labels.
Any LaTeX installation should work, as long as all the required packages are installed, e.g., [MiKTeX](https://miktex.org/) or [TeXLive](https://www.tug.org/texlive/).

If you do not have LaTeX installed, you can comment the next cell out or skip it.)

In [ ]:
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

---

**Throught the whole notebook, we will assume that the rotation speed $\omega$ can be set arbitrarily from an external load machine.**

### Operating point selection for the nonlinear PMSM:

We will use an adapted implementation from inb03 as our plant to be controlled (code in `utils/pmsm.py`) and the current controller from inb04 (code in `utils/current_controller.py`).

Now we want to control the system to produce a specific torque $T$.

We are looking at the operating point selection (OPS) component of our complete control structure:

<div>
<img src="fig/OPS_structure_diagramm.png" width="1000"/>
</div>

In [ ]:
# starting with this notebook, all auxiliary simulation code for the PMSM will be placed in 
# `CTPD/interactive_notebooks/utils/`
import sys

from path_helper import get_folder_path
sys.path.insert(1, str(get_folder_path()))

In [ ]:
from utils.pmsm import PMSM, lut_interpolate, clip_voltage
from utils.motor_params import MotorVariant
from utils.visualization import visualize_trajectories, visualize_trajectories_with_reference
from utils.signals import aprbs
from utils.current_controller import FOCController

Below we initialize the PMSM simulator and the current controller that you already know.

In [ ]:
pmsm = PMSM(
    saturated=True,
    motor_variant=MotorVariant.BRUSA,
    T_s=1e-4,
)
pmsm

In [ ]:
current_controller = FOCController(
    static_params=pmsm.env_properties.static_params,
    lut_grids=pmsm.LUT_grids,
    lut_values=pmsm.LUT_values,
    T_s=pmsm.T_s,
    d=0.99,
    rho=0.25,
) 
current_controller

Now we want to build a structure that computes target currents based on a target torque.
The plan is to build a real-time capabale mapping $T^* \rightarrow i^*$ in the form of look-up-tables (LUTs).

The structure of the LUT-based OPS is:

<div>
<img src="fig/LUT-based_OPS.jpg" width="1000"/>
</div>

We need to compute 3 LUTs to be able to use this.

The `flux_MTPC_LUT`:
$$ \large \psi^*_{\mathrm{MTPC}} (T^*) = \| \boldsymbol{\psi}_{\mathrm{s, dq}} (i^*_{\mathrm{MTPC}} (T^*))\|$$
which yields the required flux magnitude for the requested torque.

The `torque_MTPV_LUT`:
$$ \large T_{\mathrm{lim}} (\psi_{\mathrm{lim}}) = \max_{\boldsymbol{i}} T(\boldsymbol{i})$$
which yields maximum achievable torque for a given $\psi_{\mathrm{lim}} \approx u_{\mathrm{max}} / \omega$.

The `flux_torque_to_current_LUT`:
$$ \large \begin{bmatrix} i^*_{\mathrm{s,d}} & i^*_{\mathrm{s,q}} \end{bmatrix} = f(\psi^*_{\mathrm{lim}}, T^*_{\mathrm{lim}})$$
which yields the current operating point in the feasible operating region.

#### Visualize torque isolines:

In [ ]:
i_d, i_q = pmsm.LUT_grids["L_dd"]

# we create more samples (where the parameters are interpolated) to create
# a denser grid. This allows us to have a pretty accurate grid level approximation
i_d = jnp.linspace(i_d[0], i_d[-1], i_d.shape[0] * 2)
i_q = jnp.linspace(i_q[0], i_q[-1], i_q.shape[0] * 2)
i_d, i_q = jnp.meshgrid(i_d, i_q)

print(i_d.shape)
print(i_q.shape)

# double vmap to vectorize over both axes of the current input arrays
torques = eqx.filter_vmap(eqx.filter_vmap(pmsm.currents_to_torque_saturated))(i_d, i_q)
print(torques.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$i_\mathrm{d}$")
ax.set_ylabel(r"$i_\mathrm{q}$")

scatter = ax.scatter(i_d, i_q, s=0.5, label=r"$i_\mathrm{s, dq}$ grid")

ax.contour(
    i_d, i_q, jnp.sqrt(i_d**2 + i_q**2), levels=[pmsm.env_properties.static_params.i_lim], colors="red", linewidths=1.5,
)
ax.contour(i_d, i_q, torques, linestyles="dashed", colors="k")

# Create proxy artists for the legend
lim_proxy = mlines.Line2D([], [], color="red", linewidth=1.5, label=r"$i_\mathrm{lim}$")
torque_proxy = mlines.Line2D([], [], color="k", linestyle="dashed", label="torque isolines")

# Add legend using proxy artists
ax.legend(
    handles=[scatter, lim_proxy, torque_proxy],
    bbox_to_anchor=[1.9, 0.9]
)

ax.set_ylim((-250, 250))
ax.set_xlim((-250, 10))

ax.set_aspect("equal")

The black, dashed lines show the torque isolines (i.e., along the lines the torque is constant), in red we can see the current limit.

#### LUT computation:

In [ ]:
class LUT(eqx.Module):
    grid: jax.Array
    values: jax.Array

    def interpolate(x):
        raise NotImplementedError()

##### `flux_MTPC_LUT`:  $\quad \large \psi^*_\mathrm{MTPC}(T^*) $

In [ ]:
def objective(i_dq):
    return jnp.sum(i_dq**2)

def constraint(i_dq, torque_target, pmsm):
    torque_est = pmsm.currents_to_torque_saturated(i_dq[0], i_dq[1])
    return jnp.squeeze(torque_target - torque_est)

def augmented_lagrangian(i_dq, lam, rho, torque_target, pmsm):
    h = constraint(i_dq, torque_target, pmsm)
    return objective(i_dq) + lam * h + (rho / 2.0) * h**2

@eqx.filter_jit
def optimize_single(torque_target, i_dq_init):
    def al_step(i, state):
        i_dq, lam, rho, prev_viol = state

        solver = jaxopt.LBFGS(
            fun=lambda i_dq: augmented_lagrangian(i_dq, lam, rho, torque_target, pmsm),
            maxiter=300, tol=1e-7,
        )
        i_dq = solver.run(i_dq).params

        viol = constraint(i_dq, torque_target, pmsm)
        rho  = jnp.where(jnp.abs(viol) > 0.25 * jnp.abs(prev_viol), jnp.minimum(rho * 1.5, 1e2), rho)
        lam  = jnp.clip(lam + rho * viol, -1e3, 1e3)
        return i_dq, lam, rho, viol

    init_state = (i_dq_init, 0.0, 1.0, jnp.inf)
    i_dq, lam, rho, viol = jax.lax.fori_loop(0, 40, al_step, init_state)
    return i_dq

@eqx.filter_jit
def get_flux(i_dq, pmsm):
    p_d = {q: lut_interpolate(*pmsm.LUT_grids[q], pmsm.LUT_values[q], *i_dq) for q in pmsm.LUT_grids}
    psi_dq = jnp.column_stack([p_d[q] for q in ["Psi_d", "Psi_q"]])
    
    return jnp.sqrt(jnp.sum(psi_dq**2))

In [ ]:
N_T = 61
T_lut = jnp.linspace(jnp.min(torques) * 0.9, jnp.max(torques) * 0.9, N_T)

torque_targets = T_lut
i_dq_inits = jnp.ones((T_lut.shape[0], 2)) * 0.1

optimize_batch = eqx.filter_vmap(optimize_single)

In [ ]:
i_dq_solutions = optimize_batch(torque_targets, i_dq_inits)
mask_valid = jnp.sqrt(jnp.sum(i_dq_solutions**2, axis=-1)) <= pmsm.env_properties.static_params.i_lim

In [ ]:
i_dq_valid = i_dq_solutions[mask_valid]
T_lut = T_lut[mask_valid]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$i_\mathrm{d}$")
ax.set_ylabel(r"$i_\mathrm{q}$")

line = ax.plot(i_dq_valid[:, 0], i_dq_valid[:, 1], label=r"$i_\mathrm{s, dq}$ grid")

ax.contour(
    i_d, i_q, jnp.sqrt(i_d**2 + i_q**2), levels=[pmsm.env_properties.static_params.i_lim], colors="red", linewidths=1.5,
)
ax.contour(i_d, i_q, torques, linestyles="dashed", colors="k")

# Create proxy artists for the legend
lim_proxy = mlines.Line2D([], [], color="red", linewidth=1.5, label=r"$i_\mathrm{lim}$")
torque_proxy = mlines.Line2D([], [], color="k", linestyle="dashed", label="torque isolines")

# Add legend using proxy artists
ax.legend(
    handles=[lim_proxy, torque_proxy],
    bbox_to_anchor=[1.9, 0.9]
)

ax.set_ylim((-250, 250))
ax.set_xlim((-250, 10))

ax.set_aspect("equal")

Compute the flux values from the optimal currents:

In [ ]:
psi_MTPC_lut = eqx.filter_vmap(get_flux, in_axes=(0, None))(i_dq_valid, pmsm)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$T^*$")
ax.set_ylabel(r"$\psi^*_\mathrm{MTPC}$")

plt.plot(T_lut, psi_MTPC_lut)

In [ ]:
i_dq_valid_MTPC = i_dq_valid

In [ ]:
flux_MTPC_LUT = LUT(grid=T_lut, values=psi_MTPC_lut)
flux_MTPC_LUT

##### `torque_MTPV_LUT`: $\quad \large T_\mathrm{lim} (\psi^*_\mathrm{lim})$

In [ ]:
def objective(i_dq, pmsm, positive_torques):
    torque = pmsm.currents_to_torque_saturated(*i_dq)

    if positive_torques:
        return - torque  # maximize the torque, so minimize negative torque
    else:
        return torque # minimize the torque (negative values)

def constraints(i_dq, psi_lim, pmsm):

    i_lim = pmsm.env_properties.static_params.i_lim
    
    psi = get_flux(i_dq, pmsm)
    h1 = (psi - psi_lim) * 1e3
    h2 = jnp.linalg.norm(i_dq) - i_lim

    return jnp.array([h1, h2])

def augmented_lagrangian(i_dq, psi_lim, lam, rho, pmsm, positive_torques):
    h = constraints(i_dq, psi_lim, pmsm)
    al_ineq = jnp.where(
        lam + rho * h > 0,
        lam * h + (rho / 2.0) * h**2,   # active: penalize
        -lam**2 / (2.0 * rho),           # inactive: constant
    )
    return objective(i_dq, pmsm, positive_torques) + jnp.sum(al_ineq)

@eqx.filter_jit
def optimize_single(psi_lim, i_dq_init, pmsm, positive_torques=True):
    def al_step(i, state):
        i_dq, lam, rho, prev_viol = state

        solver = jaxopt.LBFGS(
            fun=lambda i_dq: augmented_lagrangian(i_dq, psi_lim, lam, rho, pmsm, positive_torques),
            maxiter=300, tol=1e-7,
        )
        i_dq = solver.run(i_dq).params

        h = constraints(i_dq, psi_lim, pmsm)
        viol = jnp.linalg.norm(jnp.maximum(h, 0.0))
        rho = jnp.where(viol > 0.25 * prev_viol, jnp.minimum(rho * 1.5, 1e2), rho)
    
        lam = jnp.maximum(lam + rho * h, 0.0)
        lam = lam * (h >= 0).astype(jnp.float32)
        return i_dq, lam, rho, viol

    init_state = (i_dq_init, jnp.zeros(2), 1.0, jnp.inf)
    i_dq, lam, rho, viol = jax.lax.fori_loop(0, 40, al_step, init_state)
    return i_dq

In [ ]:
N_psi = 201

psi_max = jnp.max(jnp.sqrt(pmsm.LUT_values["Psi_d"]**2 + pmsm.LUT_values["Psi_q"]**2))

psi_LUT = jnp.linspace(0, 0.99 * psi_max, N_psi)
i_dq_inits = jnp.ones((psi_LUT.shape[0], 2)) * jnp.array([-100, 100])

psi_limits = psi_LUT
optimize_batch = eqx.filter_vmap(optimize_single, in_axes=(0,0,None,None))

#i_dq_solutions = optimize_single(psi_limits[100], i_dq_inits[100], pmsm)

i_dq_solutions_positive_torque = optimize_batch(psi_limits, i_dq_inits, pmsm, True)
mask_valid_positive_torque = jnp.sqrt(jnp.sum(i_dq_solutions_positive_torque**2, axis=-1)) <= pmsm.env_properties.static_params.i_lim

In [ ]:
N_psi = 201

psi_max = jnp.max(jnp.sqrt(pmsm.LUT_values["Psi_d"]**2 + pmsm.LUT_values["Psi_q"]**2))

psi_LUT = jnp.linspace(0, 0.99 * psi_max, N_psi)
i_dq_inits = jnp.ones((psi_LUT.shape[0], 2)) * jnp.array([-100, 100])

psi_limits = psi_LUT
optimize_batch = eqx.filter_vmap(optimize_single, in_axes=(0,0,None,None))

#i_dq_solutions = optimize_single(psi_limits[100], i_dq_inits[100], pmsm)

i_dq_solutions_negative_torque = optimize_batch(psi_limits, i_dq_inits, pmsm, False)
mask_valid_negative_torque = jnp.sqrt(jnp.sum(i_dq_solutions_negative_torque**2, axis=-1)) <= pmsm.env_properties.static_params.i_lim

In [ ]:
psi_MTPV_lut = psi_LUT[mask_valid_positive_torque]

In [ ]:
i_dq_valid = jnp.concatenate([i_dq_solutions_negative_torque[mask_valid_negative_torque][::-1], i_dq_solutions_positive_torque[mask_valid_positive_torque]])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$i_\mathrm{d}$")
ax.set_ylabel(r"$i_\mathrm{q}$")

line_MTPC = ax.plot(i_dq_valid_MTPC[:, 0], i_dq_valid_MTPC[:, 1], color="tab:orange")
ax.contour(i_d, i_q, jnp.sqrt(i_d**2 + i_q**2), levels=[pmsm.env_properties.static_params.i_lim], colors="red", linewidths=1.5)
line_MC_MTPV = ax.plot(i_dq_valid[:, 0], i_dq_valid[:, 1], color="tab:blue")
ax.contour(i_d, i_q, torques, linestyles="dashed", colors="k")

# Create proxy artists for the legend
line_MTPC_proxy = mlines.Line2D([], [], color="tab:orange", linewidth=1.5, label="MTPC")
line_MC_MTPV_proxy = mlines.Line2D([], [], color="tab:blue", linewidth=1.5, label="MC + MTPV")
lim_proxy = mlines.Line2D([], [], color="red", linewidth=1.5, label=r"$i_\mathrm{lim}$")
torque_proxy = mlines.Line2D([], [], color="k", linestyle="dashed", label="torque isolines")

# Add legend using proxy artists
ax.legend(
    handles=[line_MTPC_proxy, line_MC_MTPV_proxy, lim_proxy, torque_proxy],
    bbox_to_anchor=[1.9, 0.9]
)


ax.set_ylim((-250, 250))
ax.set_xlim((-260, 10))

ax.set_aspect("equal")

plt.savefig("BRUSA_MTPC_MC_MTPV.pdf", bbox_inches="tight")

In [ ]:
T_MTPV_lut = eqx.filter_vmap(pmsm.currents_to_torque_saturated)(i_dq_solutions_positive_torque[mask_valid_positive_torque][:, 0], i_dq_solutions_positive_torque[mask_valid_positive_torque][:, 1])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.grid(True, alpha=0.3)
ax.set_ylabel(r"$T_\mathrm{lim}$")
ax.set_xlabel(r"$\psi_\mathrm{lim}$")

plt.plot(psi_MTPV_lut, T_MTPV_lut)

In [ ]:
torque_MTPV_LUT = LUT(grid=psi_MTPV_lut, values=T_MTPV_lut)
torque_MTPV_LUT

##### `flux_torque_to_current_LUT`: $ \large \begin{bmatrix} i^*_{\mathrm{s,d}} & i^*_{\mathrm{s,q}} \end{bmatrix} = f(\psi^*_{\mathrm{lim}}, T^*_{\mathrm{lim}})$

Get all values for $i_\mathrm{dq}$ that result in the required $T$ and $\psi$, then choose the one with the minimal amplitude.

In [ ]:
jnp.set_printoptions(linewidth=500)

In [ ]:
# from jax import config
# config.update("jax_check_tracer_leaks", True)

In [ ]:
def optimize_f_psi_T(psi_lim, T_star, id_init, pmsm):
    """1D optimization: find id on flux ellipse that gives T*"""
    
    def find_iq_for_flux(id_val):
        """Fixed-iteration Newton to find iq such that ‖ψ(id, iq)‖ = psi_lim"""
        iq = jnp.array(50.0)
        def newton_step(_, iq):
            f  = get_flux(jnp.array([id_val, iq]), pmsm) - psi_lim
            df = jax.grad(lambda iq: get_flux(jnp.array([id_val, iq]), pmsm))(iq)
            return iq - f / (df + 1e-8)
        iq = jax.lax.fori_loop(0, 100, newton_step, iq)
        return iq

    def torque_residual(id_val):
        iq = find_iq_for_flux(id_val)
        return pmsm.currents_to_torque_saturated(id_val, iq) - T_star

    def torque_residual_sq(id_val):
        return torque_residual(id_val[0])**2

    # 1D minimization over id only
    solver = jaxopt.LBFGS(fun=torque_residual_sq, maxiter=600, tol=1e-8)
    id_opt = solver.run(jnp.array([id_init])).params[0]
    iq_opt = find_iq_for_flux(id_opt)

    return jnp.array([id_opt, iq_opt])

In [ ]:
id_init = -100.0

N_psi = 10
N_T = 10

u_max = pmsm.env_properties.static_params.u_dc / jnp.sqrt(3)
n_max = 11_000
omega_el_max = omega = jnp.array(pmsm.env_properties.static_params.p * n_max * 2 * jnp.pi / 60)

psi_max = u_max / omega_el_max

psi_values = jnp.linspace(0, 0.99 * psi_max, N_psi)
T_values = jnp.linspace(jnp.min(torques) * 0.5, jnp.max(torques) * 0.5, N_T)

LUT_grid = jnp.squeeze(jnp.stack(jnp.meshgrid(psi_values, T_values), axis=-1))

In [ ]:
batched_optimize_f_psi_T = eqx.filter_vmap(
    eqx.filter_vmap(
        optimize_f_psi_T,
        in_axes=(None, 0, None, None),
    ), 
    in_axes=(0, None, None, None),
)


i_dq_solutions = eqx.filter_jit(batched_optimize_f_psi_T)(psi_values, T_values, id_init, pmsm)  # shape (n_psi, n_T, 2)

In [ ]:
resulting_flux = eqx.filter_vmap(eqx.filter_vmap(get_flux, in_axes=(0, None)), in_axes=(0, None))(i_dq_solutions, pmsm)
jnp.mean(jnp.abs(resulting_flux - LUT_grid[..., 0]))

In [ ]:
resulting_flux * 1e3

In [ ]:
LUT_grid[..., 0] * 1e3

In [ ]:
resulting_torque = eqx.filter_vmap(eqx.filter_vmap(pmsm.currents_to_torque_saturated, in_axes=(0, 0)), in_axes=(0, 0))(i_dq_solutions[..., 0], i_dq_solutions[..., 1])
jnp.mean(jnp.abs(resulting_torque - LUT_grid[..., 1]))

In [ ]:
resulting_torque

In [ ]:
LUT_grid[..., 1]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$i_\mathrm{d}$")
ax.set_ylabel(r"$i_\mathrm{q}$")

line = ax.scatter(i_dq_solutions.reshape((-1, 2))[:, 0], i_dq_solutions.reshape((-1, 2))[:, 1], color="tab:blue")
ax.contour(i_d, i_q, jnp.sqrt(i_d**2 + i_q**2), levels=[pmsm.env_properties.static_params.i_lim], colors="red", linewidths=1.5)
ax.contour(i_d, i_q, torques, linestyles="dashed", colors="k")

# Create proxy artists for the legend
line_proxy = mlines.Line2D([], [], color="blue", linewidth=1.5, label=r"$i_{\mathrm{dq}}$")
lim_proxy = mlines.Line2D([], [], color="red", linewidth=1.5, label=r"$i_\mathrm{lim}$")
torque_proxy = mlines.Line2D([], [], color="k", linestyle="dashed", label="torque isolines")

# Add legend using proxy artists
ax.legend(
    handles=[line_proxy, lim_proxy, torque_proxy],
    bbox_to_anchor=[1.9, 0.9]
)


ax.set_ylim((-400, 400))
ax.set_xlim((-600, 10))

ax.set_aspect("equal")

In [ ]:
raise

# close

In [ ]:
jnp.set_printoptions(linewidth=500)

In [ ]:
i_lim = pmsm.env_properties.static_params.i_lim

@eqx.filter_jit
def get_flux(i_dq, pmsm):
    p_d = {q: lut_interpolate(*pmsm.LUT_grids[q], pmsm.LUT_values[q], *i_dq) for q in pmsm.LUT_grids}
    psi_dq = jnp.column_stack([p_d[q] for q in ["Psi_d", "Psi_q"]])
    
    return jnp.sqrt(jnp.sum(psi_dq**2))

def objective(i_dq):
    return jnp.sum(i_dq**2)

def constraints(i_dq, torque_target, flux_target, pmsm):
    torque_est = pmsm.currents_to_torque_saturated(i_dq[0], i_dq[1])
    h1 = jnp.squeeze(torque_target - torque_est)

    flux_est = get_flux(i_dq, pmsm) 
    h2 = jnp.squeeze(flux_target - flux_est) * 1e4

    h_eq = jnp.array([h1, h2]) 
    h_ineq = jnp.linalg.norm(i_dq) - i_lim
    
    return h_eq, h_ineq

def augmented_lagrangian(i_dq, lam_eq, lam_ineq, rho, torque_target, flux_target, pmsm):
    h_eq, h_ineq = constraints(i_dq, torque_target, flux_target, pmsm)
    al_eq = jnp.sum(lam_eq * h_eq + (rho / 2.0) * h_eq**2)

    al_ineq = jnp.where(
        lam_ineq + rho * h_ineq > 0,
        lam_ineq * h_ineq + (rho / 2.0) * h_ineq**2,
        -lam_ineq**2 / (2.0 * rho),
    )
        
    return objective(i_dq) + al_eq + al_ineq

@eqx.filter_jit
def optimize_single(target, i_dq_init, pmsm):
    def al_step(i, state):
        i_dq, lam_eq, lam_ineq, rho, prev_viol = state

        torque_target = target[1]
        flux_target = target[0]
            
        solver = jaxopt.LBFGS(
            fun=lambda i_dq: augmented_lagrangian(i_dq, lam_eq, lam_ineq, rho, torque_target, flux_target, pmsm),
            maxiter=300, tol=1e-7,
        )
        i_dq = solver.run(i_dq).params

        h_eq, h_ineq = constraints(i_dq, torque_target, flux_target, pmsm)

        viol = jnp.linalg.norm(jnp.concatenate([h_eq, jnp.array([jnp.maximum(h_ineq, 0.0)])]))
        rho     = jnp.where(viol > 0.25 * prev_viol, jnp.minimum(rho * 1.5, 1e2), rho)
        lam_ineq = jnp.maximum(lam_ineq + rho * h_ineq, 0.0)
        lam_ineq = lam_ineq * (h_ineq >= 0).astype(jnp.float32)

        return i_dq, lam_eq, lam_ineq, rho, viol

    init_state = (i_dq_init, jnp.zeros(2), 0.0, 1.0, jnp.inf)
    i_dq, lam_eq, lam_ineq, rho, viol = jax.lax.fori_loop(0, 40, al_step, init_state)
    return i_dq


In [ ]:
N_psi = 5
N_T = 5

u_max = pmsm.env_properties.static_params.u_dc / jnp.sqrt(3)
n_max = 11_000
omega_el_max = omega = jnp.array(pmsm.env_properties.static_params.p * n_max * 2 * jnp.pi / 60)

psi_max = u_max / omega_el_max

psi_values = jnp.linspace(0, 0.99 * psi_max, N_psi)
T_values = jnp.linspace(jnp.min(torques) * 0.5, jnp.max(torques) * 0.5, N_T)

LUT_grid = jnp.squeeze(jnp.stack(jnp.meshgrid(psi_values, T_values), axis=-1))
LUT_grid.shape

In [ ]:
i_dq_inits = jnp.ones((LUT_grid.shape)) * jnp.array([-100.0, 100.0])

psi_limits = psi_LUT
optimize_batch = eqx.filter_vmap(eqx.filter_vmap(optimize_single, in_axes=(0,0,None)), in_axes=(0,0,None))

# i_dq_solutions = optimize_single(LUT_grid[0,0,:], i_dq_inits[0,0,:], pmsm)

i_dq_solutions = optimize_batch(LUT_grid, i_dq_inits, pmsm)

In [ ]:
# i_dq = jnp.array([-100.0, 100.0])
# i_dq_lut = []

# psi_lim_grid = LUT_grid[..., 0].flatten()
# t_lim_grid = LUT_grid[..., 1].flatten()

# for psi_lim, t_lim in zip(psi_lim_grid, t_lim_grid):
#     i_dq = optimize_single(jnp.array([t_lim, psi_lim]), i_dq, pmsm)
#     i_dq_lut.append(i_dq)

# i_dq_solutions = jnp.array(i_dq_lut).reshape(LUT_grid.shape)

In [ ]:
resulting_flux = eqx.filter_vmap(eqx.filter_vmap(get_flux, in_axes=(0, None)), in_axes=(0, None))(i_dq_solutions, pmsm)
jnp.mean(jnp.abs(resulting_flux - LUT_grid[..., 0]))

In [ ]:
resulting_flux * 1e3

In [ ]:
LUT_grid[..., 0] * 1e3

In [ ]:
resulting_torque = eqx.filter_vmap(eqx.filter_vmap(pmsm.currents_to_torque_saturated, in_axes=(0, 0)), in_axes=(0, 0))(i_dq_solutions[..., 0], i_dq_solutions[..., 1])
jnp.mean(jnp.abs(resulting_torque - LUT_grid[..., 1]))

In [ ]:
resulting_torque

In [ ]:
LUT_grid[..., 1]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.grid(True, alpha=0.3)
ax.set_xlabel(r"$i_\mathrm{d}$")
ax.set_ylabel(r"$i_\mathrm{q}$")

line = ax.scatter(i_dq_solutions.reshape((-1, 2))[:, 0], i_dq_solutions.reshape((-1, 2))[:, 1], color="tab:blue")
ax.contour(i_d, i_q, jnp.sqrt(i_d**2 + i_q**2), levels=[pmsm.env_properties.static_params.i_lim], colors="red", linewidths=1.5)
ax.contour(i_d, i_q, torques, linestyles="dashed", colors="k")

# Create proxy artists for the legend
line_proxy = mlines.Line2D([], [], color="blue", linewidth=1.5, label=r"$i_{\mathrm{dq}}$")
lim_proxy = mlines.Line2D([], [], color="red", linewidth=1.5, label=r"$i_\mathrm{lim}$")
torque_proxy = mlines.Line2D([], [], color="k", linestyle="dashed", label="torque isolines")

# Add legend using proxy artists
ax.legend(
    handles=[line_proxy, lim_proxy, torque_proxy],
    bbox_to_anchor=[1.9, 0.9]
)


ax.set_ylim((-400, 400))
ax.set_xlim((-600, 10))

ax.set_aspect("equal")

In [ ]:
flux_torque_to_current_LUT = LUT(grid=None, values=None)

- Full OPS:

In [ ]:
T_lut, psi_MTPC_lut

In [ ]:
psi_MTPV_lut, T_MTPV_lut

In [ ]:
combine the luts provide 

#### Full control structure:

In [ ]:
raise NotImplementedError()

In [ ]:
raise

### WIP stuff:

In [ ]:
def find_fitting_psi_idq(T_k, torques, i_d, i_q, i_s_max, tolerance=1e-1):
    mask = jnp.isclose(torques, T_k, atol=tolerance)
    
    if not jnp.any(mask):
        return (None, None, None)

    i_idx, j_idx = jnp.where(mask)
    i_d_candidates = i_d[i_idx, j_idx]
    i_q_candidates = i_q[i_idx, j_idx]
    
    sign_ok = (i_q_candidates >= 0) if T_k >= 0 else (i_q_candidates <= 0)
    current_ok = np.sqrt(i_d_candidates**2 + i_q_candidates**2) <= i_s_max
    ok = sign_ok & current_ok

    if not jnp.any(ok):
        ok = current_ok

    if not jnp.any(ok):
        return (None, None, None)

    # from the remaining points, take the one with the minimal current amplitude
    i_d_feasible, i_q_feasible = i_d_candidates[ok], i_q_candidates[ok]
    best = jnp.argmin(i_d_feasible**2 + i_q_feasible**2)
    i_d_lut = i_d_feasible[best]
    i_q_lut = i_q_feasible[best]

    p_d = {q: lut_interpolate(*pmsm.LUT_grids[q], pmsm.LUT_values[q], i_d_lut, i_q_lut) for q in pmsm.LUT_grids}
    psi_dq = jnp.column_stack([p_d[q] for q in ["Psi_d", "Psi_q"]])

    psi_lut = jnp.sqrt(jnp.sum(psi_dq**2))

    return i_d_lut, i_q_lut, psi_lut

In [ ]:
N_T = 41
T_lut = jnp.linspace(jnp.min(torques) * 0.95, jnp.max(torques) * 0.95, N_T)

i_d_MTPC_lut  = np.zeros(N_T)
i_q_MTPC_lut  = np.zeros(N_T)
psi_MTPC_lut = np.zeros(N_T)

i_s = jnp.sqrt(i_d**2 + i_q**2)
torques = eqx.filter_vmap(eqx.filter_vmap(pmsm.currents_to_torque_saturated))(i_d, i_q)

for k, T_k in enumerate(T_lut):
    i_d_lut, i_q_lut, psi_lut = find_fitting_psi_idq(
        T_k, torques, i_d, i_q, pmsm.env_properties.static_params.i_lim
    )

    if i_d_lut is None:
        assert i_q_lut is None
        assert psi_lut is None
        if k == 0:
            i_d_lut, i_q_lut, psi_lut = (0.0, 0.0, 0.0)
        else:
            i_d_MTPC_lut[k] = i_d_MTPC_lut[k-1]
            i_q_MTPC_lut[k] = i_q_MTPC_lut[k-1]
            psi_MTPC_lut[k] = psi_MTPC_lut[k-1]
    else:
        i_d_MTPC_lut[k] = i_d_lut
        i_q_MTPC_lut[k] = i_q_lut
        psi_MTPC_lut[k] = psi_lut

In [ ]:
T_lut

In [ ]:
plt.plot(T_LUT, psi_MTPC_lut)

In [ ]:
import jaxopt

In [ ]:
import jax.numpy as jnp
import jaxopt

def objective(i_dq):
    return jnp.sum(i_dq**2)

def constraint(i_dq, torque_target, pmsm):
    torque_est = pmsm.currents_to_torque_saturated(i_dq[0], i_dq[1])
    return jnp.squeeze(torque_target - torque_est)

def augmented_lagrangian(i_dq, lam, rho, torque_target, pmsm):
    h = constraint(i_dq, torque_target, pmsm)
    return objective(i_dq) + lam * h + (rho / 2.0) * h**2

def optimize_currents(i_dq_init, torque_target, pmsm):
    i_dq = i_dq_init
    lam  = 0.0 
    rho  = 1.0
    prev_viol = jnp.inf

    for i in range(30):
        solver = jaxopt.LBFGS(
            fun=lambda i_dq: augmented_lagrangian(i_dq, lam, rho, torque_target, pmsm),
            maxiter=200,
            tol=1e-7,
        )
        result = solver.run(i_dq)
        i_dq = result.params

        viol = constraint(i_dq, torque_target, pmsm)

        # Only increase rho if violation isn't shrinking
        if jnp.abs(viol) > 0.25 * jnp.abs(prev_viol):
            rho = min(rho * 1.2, 100)

        lam = jnp.clip(lam + rho * viol, -1e6, 1e6)
        prev_viol = viol

        print(f"iter {i:2d} | |i_dq|={jnp.sqrt(objective(i_dq)):.4f} | "
              f"torque_err={viol:.2e} | rho={rho:.1f} | lam={lam:.1f}")

        if jnp.abs(viol) < 1e-6:
            break

    return i_dq, viol

In [ ]:
N_T = 41
T_lut = jnp.linspace(jnp.min(torques) * 0.95, jnp.max(torques) * 0.95, N_T)

i_d_MTPC_lut  = np.zeros(N_T)
i_q_MTPC_lut  = np.zeros(N_T)
psi_MTPC_lut = np.zeros(N_T)

i_dq_init = jnp.array([0.1, 0.1])

for k, T_k in enumerate(T_lut):
    i_dq, final_error = optimize_currents(i_dq_init, torque_target=jnp.array([10.0]), pmsm=pmsm)

    i_d_MTPC_lut[k] = i_dq[0]
    i_q_MTPC_lut[k] = i_dq[1]

    p_d = {q: lut_interpolate(*pmsm.LUT_grids[q], pmsm.LUT_values[q], *i_dq) for q in pmsm.LUT_grids}
    psi_dq = jnp.column_stack([p_d[q] for q in ["Psi_d", "Psi_q"]])

    psi_MTPC_lut[k] = jnp.sqrt(jnp.sum(psi_dq**2))
    
    
    i_dq_init = i_dq
